# Exercise — Adjudicate Master-Data Matches & Set Survivorship

**Trailhead Provisions** has duplicate customers because marketing keys on email while loyalty
keys on `customer_id`. Resolution auto-merges the obvious (`score >= 0.85`) and rejects the obvious
(`score < 0.66`); the **ambiguous** band (`0.66–0.84` — *pairs that share a name and an email that
differs only by mail provider or a trailing digit*) needs your judgment. Run the resolution workflow,
adjudicate all **8** pairs in the ambiguous band (`0..7`), pass your survivorship rule, and document
your stewardship log. See `INSTRUCTIONS.md`.

## Step 0: Connect to your Cloud Lab

Run the cell below to configure your live AWS environment and ensure the `trailhead` Glue Data Catalog and S3 tables are seeded in your lab account.

In [1]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
# Seed the Glue Data Catalog and S3 tables if not yet present in the lab account
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

In [2]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend, apply_adjudications

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__} (Glue DB: {gc.database}, Region: {gc.region})")

m = gc.resolve_customers()
print("auto-merge:", int((m.pairs.band == "auto-merge").sum()),
      "| ambiguous:", len(m.ambiguous),
      "| rejected:", int((m.pairs.band == "reject").sum()))
# Display all signal columns (phone, country, created_ts) to separate true duplicates from homonyms.
# You can also inspect full customer rows with gc.query("SELECT * FROM customer WHERE customer_id IN (...)").
m.ambiguous[["id_a", "name_a", "email_a", "phone_a", "country_a", "created_a",
             "id_b", "name_b", "email_b", "phone_b", "country_b", "created_b", "score"]]

Backend: live -> AwsGovernedCatalog (Glue DB: trailhead, Region: us-east-1)
auto-merge: 2 | ambiguous: 8 | rejected: 0


,id_a,name_a,email_a,phone_a,country_a,created_a,id_b,name_b,email_b,phone_b,country_b,created_b,score
0,45,Esme Reyes,esme.reyes@trailmail.co,+1058040488,CA,2017-08-20,307,Esme Reyes,esme.reyes@outlook.com,+1058040488,CA,2017-08-20,0.80
1,133,Esme Sattar,esme.sattar@gmail.com,+81084471562,IN,2023-08-30,306,Esme Sattar,esme.sattar2@gmail.com,+81084471562,IN,2023-08-30,0.80
2,159,Yara Pope,yara.pope@proton.me,+49578658551,DE,2014-12-06,303,Yara Pope,yara.pope@outlook.com,+49578658551,DE,2014-12-06,0.80
3,190,Felix Reyes,felix.reyes@trailmail.co,+55952523669,MX,2024-11-05,308,Felix Reyes,felix.reyes6@trailmail.co,+55952523669,MX,2024-11-05,0.80
4,249,Esme Saleh,esme.saleh@trailmail.co,,BR,2024-03-29,304,Esme Saleh,esme.saleh5@trailmail.co,,BR,2024-03-29,0.80
5,272,Yara Adeyemi,yara.adeyemi@outlook.com,+49296788738,DE,2024-08-23,305,Yara Adeyemi,yara.adeyemi@gmail.com,+49296788738,DE,2024-08-23,0.80
6,1,Zane Saleh,zane.saleh@proton.me,+81936622768,IN,2024-12-12,309,Zane Saleh,zane.saleh2@outlook.com,+14158902341,US,2022-03-14,0.75
7,3,Pia Lindqvist,pia.lindqvist@outlook.com,+1919729372,CA,2023-11-18,310,Pia Lindqvist,pia.lindqvist7@proton.me,+55119876543,BR,2021-05-11,0.75


## 1. Adjudicate each ambiguous pair & set survivorship
Decide `"accept"` (same person) or `"reject"` (different people / homonym) for every row index `0..7` in `m.ambiguous`, and specify your `survivor` rule (`"lowest_id"` for the loyalty system of record, or `"earliest_created"`). Inspect the signal columns (`phone`, `country`, `created_a`/`created_b`) above before deciding.

In [3]:
# Per-pair adjudication across all 8 ambiguous pairs (0..7):
# - Row 0 (45 <-> 307, Esme Reyes):   same name + domain-swapped email (trailmail.co -> outlook.com), same phone/country/created_ts -> accept
# - Row 1 (133 <-> 306, Esme Sattar): same name + trailing-digit email (esme.sattar2@gmail.com), same phone/country/created_ts -> accept
# - Row 2 (159 <-> 303, Yara Pope):   same name + domain-swapped email (proton.me -> outlook.com), same phone/country/created_ts -> accept
# - Row 3 (190 <-> 308, Felix Reyes): same name + trailing-digit email (felix.reyes6@trailmail.co), same phone/country/created_ts -> accept
# - Row 4 (249 <-> 304, Esme Saleh):  same name + trailing-digit email (esme.saleh5@trailmail.co), both missing phone, same country (BR) & created_ts -> accept
# - Row 5 (272 <-> 305, Yara Adeyemi): same name (extra space) + domain-swapped email (outlook.com -> gmail.com), same phone/country/created_ts -> accept
#   (Note: all 6 accepted pairs have an email variant — domain swap or trailing digit — corroborated by matching country and created_ts.)
# - Row 6 (1 <-> 309, Zane Saleh):    homonym non-match — different phone (+81936622768 vs +14158902341), country (IN vs US), created_ts -> reject
# - Row 7 (3 <-> 310, Pia Lindqvist): homonym non-match — different phone (+1919729372 vs +55119876543), country (CA vs BR), created_ts -> reject
decisions = {
    0: "accept", 1: "accept", 2: "accept",
    3: "accept", 4: "accept", 5: "accept",
    6: "reject", 7: "reject",
}
survivor = "lowest_id"
outcome = apply_adjudications(m, decisions, survivor=survivor)
print({k: v for k, v in outcome.items() if k != "merges"})
outcome["merges"]

{'ambiguous_pairs': 8, 'accepted_merges': 6, 'rejected': 2, 'survivorship_rule': 'lowest customer_id wins (loyalty system of record)', 'golden_records_after': 'merged 6 duplicate(s)'}


,golden_id,merged_id,survivorship
0,45,307,lowest customer_id wins (loyalty system of rec...
1,133,306,lowest customer_id wins (loyalty system of rec...
2,159,303,lowest customer_id wins (loyalty system of rec...
3,190,308,lowest customer_id wins (loyalty system of rec...
4,249,304,lowest customer_id wins (loyalty system of rec...
5,272,305,lowest customer_id wins (loyalty system of rec...


## 2. Stewardship decision log
Replace the cell below with your log.

### Stewardship decision log

| Row | Pair (`id_a ↔ id_b`) | Name | Signal | Decision | Rationale |
|---|---|---|---|---|---|
| 0 | `45 ↔ 307` | Esme Reyes | same name + domain-swapped email (`trailmail.co` → `outlook.com`), same phone (`+1058040488`), country (`CA`), `created_ts` (`2017-08-20`) | **accept** | same person re-registered with a different mail provider |
| 1 | `133 ↔ 306` | Esme Sattar | same name + trailing-digit email (`esme.sattar` → `esme.sattar2@gmail.com`), same phone (`+81084471562`), country (`IN`), `created_ts` (`2023-08-30`) | **accept** | typo/auto-suffix on re-entry; identical phone, country, and timestamp |
| 2 | `159 ↔ 303` | Yara Pope | same name + domain-swapped email (`proton.me` → `outlook.com`), same phone (`+49578658551`), country (`DE`), `created_ts` (`2014-12-06`) | **accept** | same person re-registered with a different mail provider |
| 3 | `190 ↔ 308` | Felix Reyes | same name + trailing-digit email (`felix.reyes` → `felix.reyes6@trailmail.co`), same phone (`+55952523669`), country (`MX`), `created_ts` (`2024-11-05`) | **accept** | auto-suffixed email handle with identical phone, country, and timestamp |
| 4 | `249 ↔ 304` | Esme Saleh | same name + trailing-digit email (`esme.saleh` → `esme.saleh5@trailmail.co`), both missing phone, same country (`BR`) and `created_ts` (`2024-03-29`) | **accept** | same person re-registered on the same date in `BR` with a trailing digit on the email local-part |
| 5 | `272 ↔ 305` | Yara Adeyemi | same name (extra space) + domain-swapped email (`outlook.com` → `gmail.com`), same phone/country/`created_ts` (`+49296788738`, `DE`, `2024-08-23`) | **accept** | same name (extra space) + domain-swapped email (`outlook.com` → `gmail.com`), same phone/country/`created_ts` → accept (all 6 accepted pairs have an email variant) |
| 6 | `1 ↔ 309` | Zane Saleh | same name + trailing-digit/domain-swapped email (`zane.saleh@proton.me` vs `zane.saleh2@outlook.com`), **different** phone (`+81936622768` vs `+14158902341`), country (`IN` vs `US`), `created_ts` (`2024-12-12` vs `2022-03-14`) | **reject** | homonym collision across two different countries (`IN` vs `US`), phones, and registration years |
| 7 | `3 ↔ 310` | Pia Lindqvist | same name + trailing-digit/domain-swapped email (`pia.lindqvist@outlook.com` vs `pia.lindqvist7@proton.me`), **different** phone (`+1919729372` vs `+55119876543`), country (`CA` vs `BR`), `created_ts` (`2023-11-18` vs `2021-05-11`) | **reject** | homonym collision across `CA` and `BR` with distinct phones and registration dates |

**Survivorship rule (`survivor="lowest_id"`):** The **lowest `customer_id` wins** as the golden
record — **Customer Identity & Loyalty** is the designated customer system of record and lower IDs
are the original registrations; non-key attributes take the most-recent non-null value.

**False-positive vs false-negative tradeoff:** We accept only pairs where the name and email-base
match is corroborated by phone, country, and registration timestamp (`rows 0..5`), and we **reject**
homonym pairs whose phone, country, and `created_ts` disagree (`rows 6..7`: `Zane Saleh` in `IN` vs
`US`, `Pia Lindqvist` in `CA` vs `BR`). Under GDPR, a **false positive** (over-merging two different
people into one golden record) is far worse than a **false negative** (leaving a duplicate unmerged):
over-merging leaks one customer's PII, order history, and consent state to another person (a
cross-customer data breach under GDPR Art. 5(1)(f) and Art. 32), whereas a missed duplicate only
leaves two records fragmented until more signal arrives.

**Structural cross-domain fix:** Push the loyalty `customer_id` into Marketing (`campaign_membership`)
as the mandatory shared foreign key at capture time, eliminating email-based resolution entirely.

---

# Solution — Adjudicate Master Data

## Summary

The skill is **stewardship judgment on the ambiguous band** (*pairs that share a name and an email
that differs only by mail provider or a trailing digit*): accepting true duplicates corroborated by
secondary signals (phone, country, `created_ts`), rejecting homonym non-matches, setting an explicit
survivorship rule, and reasoning about the cost of over- vs under-merging under GDPR.

## Key steps

Of the **8** ambiguous pairs (`0..7`), rows `0..5` (`score == 0.80`) are true duplicates (each has
an email variant — domain swap or trailing digit — plus matching phone, country, and `created_ts`),
while rows `6..7` (`score == 0.75`) are homonym non-matches (different people who share a name and
email base, but have different phones, countries, and registration dates):

```python
decisions = {
    0: "accept", 1: "accept", 2: "accept",
    3: "accept", 4: "accept", 5: "accept",
    6: "reject", 7: "reject",
}
outcome = apply_adjudications(m, decisions, survivor="lowest_id")
```

### Per-pair stewardship decision log (all 8 pairs)

| Row | Pair (`id_a ↔ id_b`) | Name | Signal | Decision | Rationale |
|---|---|---|---|---|---|
| 0 | `45 ↔ 307` | Esme Reyes | same name + domain-swapped email (`trailmail.co` → `outlook.com`), same phone (`+1058040488`), country (`CA`), `created_ts` (`2017-08-20`) | **accept** | same person re-registered with a different mail provider |
| 1 | `133 ↔ 306` | Esme Sattar | same name + trailing-digit email (`esme.sattar` → `esme.sattar2@gmail.com`), same phone (`+81084471562`), country (`IN`), `created_ts` (`2023-08-30`) | **accept** | typo/auto-suffix on re-entry; identical phone, country, and timestamp |
| 2 | `159 ↔ 303` | Yara Pope | same name + domain-swapped email (`proton.me` → `outlook.com`), same phone (`+49578658551`), country (`DE`), `created_ts` (`2014-12-06`) | **accept** | same person re-registered with a different mail provider |
| 3 | `190 ↔ 308` | Felix Reyes | same name + trailing-digit email (`felix.reyes` → `felix.reyes6@trailmail.co`), same phone (`+55952523669`), country (`MX`), `created_ts` (`2024-11-05`) | **accept** | auto-suffixed email handle with identical phone, country, and timestamp |
| 4 | `249 ↔ 304` | Esme Saleh | same name + trailing-digit email (`esme.saleh` → `esme.saleh5@trailmail.co`), both missing phone, same country (`BR`) and `created_ts` (`2024-03-29`) | **accept** | same person re-registered on the same date in `BR` with a trailing digit on the email local-part |
| 5 | `272 ↔ 305` | Yara Adeyemi | same name (extra space) + domain-swapped email (`outlook.com` → `gmail.com`), same phone/country/`created_ts` (`+49296788738`, `DE`, `2024-08-23`) | **accept** | same name (extra space) + domain-swapped email (`outlook.com` → `gmail.com`), same phone/country/`created_ts` → accept (all 6 accepted pairs have an email variant) |
| 6 | `1 ↔ 309` | Zane Saleh | same name + trailing-digit/domain-swapped email (`zane.saleh@proton.me` vs `zane.saleh2@outlook.com`), **different** phone (`+81936622768` vs `+14158902341`), country (`IN` vs `US`), `created_ts` (`2024-12-12` vs `2022-03-14`) | **reject** | homonym collision across two different countries, phones, and registration years |
| 7 | `3 ↔ 310` | Pia Lindqvist | same name + trailing-digit/domain-swapped email (`pia.lindqvist@outlook.com` vs `pia.lindqvist7@proton.me`), **different** phone (`+1919729372` vs `+55119876543`), country (`CA` vs `BR`), `created_ts` (`2023-11-18` vs `2021-05-11`) | **reject** | homonym collision across `CA` and `BR` with distinct phones and registration dates |

*(Note: in `m.ambiguous`, pair `272 ↔ 305` is at row index `5` and `249 ↔ 304` is at row index `4`; whether ordered by customer ID or score, all 6 accepted pairs `0..5` share an email variant plus corroborating phone/country/`created_ts`, while rows `6..7` are rejected as homonyms.)*

**Survivorship rule:** `survivor="lowest_id"` — lowest `customer_id` wins as the golden record
because Customer Identity & Loyalty is the designated system of record and lower IDs represent the
original registration; non-key attributes take the most-recent non-null value.

**False-positive vs false-negative tradeoff:** Under GDPR, a **false positive** (over-merging two
different people, such as the `Zane Saleh` or `Pia Lindqvist` homonyms in rows `6` and `7`) is far
worse than a **false negative** (leaving a true duplicate unmerged): a false merge contaminates one
person's profile with another's PII and consent state (a cross-customer data breach under GDPR Art.
5(1)(f) & Art. 32), whereas a missed duplicate only causes temporary fragmentation.

**Structural fix:** Push the loyalty `customer_id` into Marketing (`campaign_membership`) as the
mandatory foreign key at capture time, eliminating email-based cross-domain resolution entirely.

## Common Mistakes to Avoid

- **Rubber-stamping all 8 ambiguous pairs as `"accept"`** without checking phone, country, and
  `created_ts` — rows `6` (`Zane Saleh`, `IN` vs `US`) and `7` (`Pia Lindqvist`, `CA` vs `BR`) are
  homonyms that must be rejected.
- **Describing `272 ↔ 305` (`Yara Adeyemi`) as only a whitespace difference** — it also has a
  domain-swapped email (`outlook.com` → `gmail.com`); all 6 accepted pairs have an email variant.
- **No survivorship rule** (or omitting `survivor=...` / not tying it to the loyalty system of record).
- **Ignoring the FP/FN asymmetry** — over-merging is the worse GDPR error.
- **Stopping at the merge** without naming the structural shared-key fix.